# IAB 網路拓撲：建立與查看部署結果

## 這份文件可以做什麼？

- 建立一張位置固定的網路部署圖，查看 Node、Tag 的位置與連接關係。
- 查看每個 Tag 屬於哪個 Node、距離多遠，以及位於近區、中區還是遠區。
- 將同一份部署結果整理成表格，並儲存圖片與資料檔。

## 目前的配置

- **Donor：** 1 個上層節點，直接連接全部 5 個 IAB Node。
- **Node：** 每個節點連接 1 個 UE，以及至少 2 個 Tag。
- **Tag：** 預設共 30 個，每個 Tag 只屬於一個 Node。
- **位置：** 只考慮平面座標，單位為公尺；一次模擬中位置不移動。

## 第一次如何操作？

1. 選擇可執行 Python，並已具備 Matplotlib 與 IPython 的環境。
2. 先保留預設設定，從頭執行全部程式區塊。
3. 查看四份表格、檢查訊息與最後的部署圖。
4. 需要換一組隨機配置時，再修改第 6 節的 `seed`。

## 使用時要知道的限制

- 目前呈現位置與連接關係，尚未計算訊號品質、傳輸成功率、電量或排程。
- 圖上顏色表示距離區域；部署圓域分離，也不代表無線訊號一定沒有干擾。
- UE 只記錄所屬 Node，沒有設定實際位置，因此不畫在圖上。
- Node 整合 Tag 讀取功能是本研究的設定，在 IAB 分離式基地台架構中的整合屬研究擴充。
- 執行最後的匯出區塊會覆寫輸出資料夾內的同名檔案；保留舊結果前請先複製到其他資料夾。

- 完整操作與常見問題請參考 [IAB 網路拓撲使用說明](../ReferDoc/Implementation/IABTopology/README.md)。


## 1. 準備設定與資料格式

- 請先執行下方第一個程式區塊，再執行後續匯入。
- 程式會向上尋找 `src/iab`，並將 `src` 加入 Python 搜尋路徑。
- 支援從專案根目錄、`notebooks/` 或其他子目錄啟動。
- 找不到時，請從專案內重新啟動 Notebook。
- 下方程式載入本次使用的資料格式與範例設定，不會在這一步產生 Tag。
- `Position2D`：記錄 x、y 座標，單位為公尺。
- `FixedNodeConfig`：記錄 Donor 或 Node 的名稱與位置。
- `TopologyConfig`：集中本次設定，包括站點位置、區域半徑、Tag 數量與 `seed`。
- `UERecord`、`TagRecord`：分別記錄 UE、Tag 的資料與所屬 Node。
- `TopologySnapshot`：保存本次設定及全部 UE、Tag 結果；後續表格與圖片都使用這一份結果。

### 填寫設定時請注意

- Node 清單、三區半徑、三區 Tag 數量都使用小括號 `(...)`，例如 `(10, 13, 7)`。
- 不接受中括號 `[...]`，程式也不會自動轉換。
- 設定與產生的資料受到保護，不能直接修改其中的欄位；需要新配置時，先建立新設定再重新生成。
- `example_config()` 提供範例座標，方便第一次操作；這些位置不是實際農場的量測資料。

### 程式與測試放在哪裡？

- `src/iab/contracts.py`：共用資料格式。
- `src/iab/topology.py`：檢查設定、生成結果與匯出。
- `src/iab/plotting.py`：繪製部署圖。
- `tests/iab/`：此模組的情境檢查與 Notebook 執行工具。


In [1]:
from pathlib import Path
import sys
import math
import random
from collections import Counter
from dataclasses import asdict

# 從目前位置向上找專案，讓 Notebook 可直接使用 src 裡的模組。
current_dir = Path.cwd().resolve()
for project_root in (current_dir, *current_dir.parents):
    if (project_root / 'src/iab/topology.py').is_file():
        break
else:
    raise RuntimeError('找不到 src/iab；請從專案根目錄或其子目錄啟動 Notebook')

source_root = str(project_root / 'src')
if source_root not in sys.path:
    sys.path.insert(0, source_root)

from iab.contracts import (Position2D, FixedNodeConfig, TopologyConfig,
                           UERecord, TagRecord, TopologySnapshot, TagZone)
from iab.topology import example_config


## 2. 距離怎麼分區？哪些設定會被拒絕？

### 三區的判定方式

- 距離都是從 Tag 所屬的 Node 開始量，使用平面直線距離。
- **近區：** 大於 0 m，且不超過 10 m。
- **中區：** 大於 10 m，且不超過 30 m。
- **遠區：** 大於 30 m，且不超過 50 m。
- 例如距離 10 m 屬近區，20 m 屬中區，30 m 仍屬中區。
- Tag 不可與所屬 Node 的圓心重合，也不可超過最外圈半徑。

### 常見設定限制

- 必須有 5 個 Node，每個至少分到 2 個 Tag，因此預設最低總數為 10 個。
- 三個區域半徑必須是正數，並由小到大排列。
- Donor 與 Node 名稱不能空白、重複，或使用會與 UE、Tag 撞名的編號。
- 座標必須是有效數值；負座標可以使用。
- 預設最外圈半徑是 50 m，因此任兩 Node 圓心距離必須大於 100 m；重疊與相切都拒絕。
- 不符合設定時，程式會顯示 `ValueError` 與原因；依訊息修改設定後再執行。

- `classify_distance()` 負責判定區域；`validate_config()` 負責檢查設定，這一步不會寫入檔案。


In [2]:
from iab.topology import classify_distance


In [3]:
from iab.topology import validate_config


## 3. Tag 如何分配與放置？

- `build_topology(config)` 讀取本次設定，產生完整部署結果。

### 產生順序

1. 檢查設定是否符合規則。
2. 依設定準備全場近／中／遠區 Tag 數量。
3. 先保留每個 Node 最少需要的 Tag 名額，再將剩餘 Tag 隨機分配給 Node。
4. 在各 Tag 所屬 Node 的指定區域內，隨機產生位置。
5. 為每個 Node 建立一個 UE 的歸屬紀錄，整理成本次部署結果。

### 如何理解隨機分配？

- 例如全場近／中／遠區為 10／13／7 個，這些總數會保留，但各 Node 分到的數量可以不同。
- 同一個 Node 可以接到多個 Tag；同一個 Tag 只會有一個所屬 Node。
- 區域內按面積隨機放置，也就是相同面積的地方有相同的抽中機會。
- `seed` 是控制隨機結果的編號；相同程式與環境下，保留設定與編號可重現相同配置。
- 拓撲生成使用自己的亂數來源，避免與其他功能互相影響。

### 距離接近邊界時

- 電腦的小數計算可能產生極微小誤差，例如應在 30 m 的位置，算出來略大於 30 m。
- 程式只在邊界附近嘗試微調位置，調整後仍須落在原本區域內。
- 若座標太大而無法準確表示距離，會顯示錯誤；請改用較接近場景實際尺度的座標。


In [4]:
from iab.topology import build_topology


## 4. 四份表格各看什麼？

- `topology_tables(snapshot)` 將本次部署結果整理成四份表格。
- **設定表 `settings`：** 查看本次位置、半徑、Tag 數量與 `seed`。
- **Node 表 `nodes`：** 查看各 Node 的位置、UE，以及近／中／遠區的 Tag 數量與總數。
- **Tag 表 `tags`：** 查看各 Tag 的編號、座標、所屬 Node、距離與區域。
- **連線表 `links`：** 查看 Donor 連到哪些 Node，各 Node 又連到哪些 UE、Tag。
- 四份表都來自同一份結果，不會因為查看表格而重新分配 Tag。
- 預設範例應有 5 條 Donor–Node 關係、5 條 Node–UE 關係，以及 30 條 Node–Tag 關係。


In [5]:
from iab.topology import topology_tables


## 5. 部署圖上的符號代表什麼？

- `plot_topology(snapshot)` 使用本次結果畫圖，不會重新產生位置。
- **星形：** Donor 的位置。
- **方形：** Node 的位置。
- **藍綠、橘褐、紫色圓框與 Tag 圓點：** 分別表示近區、中區、遠區。
- **圓點旁的數字：** Tag 編號，例如 `001` 代表 `Tag-001`。
- **Donor 與 Node 間的線：** 兩者的連接關係。
- x、y 軸都使用公尺，兩軸比例相同，圓不會因比例不同而被拉長。
- UE 沒有實際座標，因此只在表格記錄歸屬，不畫位置。
- 顏色只表示距離區域，不能直接當成訊號品質。
- 圖中使用英文名稱與編號，方便在不同電腦上顯示。

### 匯出功能

- 下方載入的 `export_topology()` 負責將結果寫成檔案；真正匯出會在最後的程式區塊執行。
- 會保存完整結果 `snapshot.json`、四份 CSV 表格，以及 PNG／SVG 部署圖。


In [6]:
from iab.plotting import plot_topology


In [7]:
from iab.topology import export_topology


## 6. 可直接修改的設定與生成結果

### 先執行預設範例

- 下方預設使用 `seed=42`。
- 全場 Tag 共 30 個：近區 10 個、中區 13 個、遠區 7 個。
- 每個 Node 至少有 2 個 Tag，各 Node 總數不必相同。
- 執行後，`config` 代表本次設定，`snapshot` 是部署結果，`tables` 是整理好的表格。

### 想換一組配置時

- 先只把 `seed=42` 改成 `seed=7`，再從頭執行全部程式區塊。
- Donor 與 Node 的指定位置不變，Tag 的分配與位置會重新產生。
- 想保留目前輸出檔案時，請先複製結果資料夾，再執行最後的匯出區塊。

### 想改 Tag 數量或區域半徑時

- 可以建立新設定，但仍須符合第 2 節的限制。
- 第 7 節的教學檢查使用預設 30 個、三區 10／13／7 個，以及 10／30／50 m 邊界。
- 若修改這些數值，也要同步調整對應檢查；只想先看新表格時，可逐段執行到表格顯示區塊。


In [8]:
config = example_config(seed=42)
snapshot = build_topology(config)
tables = topology_tables(snapshot)
print(f'生成完成：1 Donor、{len(config.iab_nodes)} Nodes、{len(snapshot.ues)} UE、{len(snapshot.tags)} Tags')


生成完成：1 Donor、5 Nodes、5 UE、30 Tags


### 查看下方表格

- 依序顯示設定表、Node 表、Tag 表與連線表。
- 先看 Node 表的各區數量，再用 Tag 表查詢個別 Tag。
- `distance_m` 表示 Tag 到所屬 Node 的距離，單位為公尺。
- `parent_node_id` 表示這個 Tag 或 UE 所屬的 Node 名稱。


In [9]:
from html import escape
from IPython.display import display, HTML, Image

for name, rows in tables.items():
    headers = list(rows[0])
    heading = {'settings': '完整設定摘要', 'nodes': 'Node數量與UE歸屬',
               'tags': 'Tag位置／距離／分區', 'links': '回傳與接入歸屬'}[name]
    content = '<h3>' + heading + '</h3><table><thead><tr>'
    content += ''.join('<th>' + escape(key) + '</th>' for key in headers)
    content += '</tr></thead><tbody>'
    for row in rows:
        content += '<tr>' + ''.join('<td>' + escape(str(row[key])) + '</td>' for key in headers) + '</tr>'
    display(HTML(content + '</tbody></table>'))


parameter,value
seed,42
zone_outer_radii_m,"(10.0, 30.0, 50.0)"
zone_tag_counts,"(10, 13, 7)"
min_tags_per_node,2
donor,"{'node_id': 'Donor-1', 'position': {'x_m': 120, 'y_m': -120}}"
iab_nodes,"[{'node_id': 'Node-1', 'position': {'x_m': 0, 'y_m': 0}}, {'node_id': 'Node-2', 'position': {'x_m': 120, 'y_m': 0}}, {'node_id': 'Node-3', 'position': {'x_m': 240, 'y_m': 0}}, {'node_id': 'Node-4', 'position': {'x_m': 60, 'y_m': 120}}, {'node_id': 'Node-5', 'position': {'x_m': 180, 'y_m': 120}}]"


node_id,x_m,y_m,donor_id,ue_id,near,mid,far,total
Node-1,0,0,Donor-1,UE-1,2,3,3,8
Node-2,120,0,Donor-1,UE-2,1,1,1,3
Node-3,240,0,Donor-1,UE-3,3,4,1,8
Node-4,60,120,Donor-1,UE-4,2,2,1,5
Node-5,180,120,Donor-1,UE-5,2,3,1,6


tag_id,parent_node_id,x_m,y_m,distance_m,zone
Tag-001,Node-5,174.1063074302299,99.91561124436129,20.931275254857027,MID
Tag-002,Node-4,64.06724480512494,149.09979403313113,29.382656331167034,MID
Tag-003,Node-3,262.67331601498324,12.426196924361742,25.855166391232387,MID
Tag-004,Node-2,157.14474611455543,27.34597626539543,46.12520549355147,FAR
Tag-005,Node-2,102.86941863752779,-19.600109685806494,26.031156668729224,MID
Tag-006,Node-1,-5.462265968595314,5.803484521502326,7.969741658485021,NEAR
Tag-007,Node-4,59.05335561742059,128.84042766701828,8.890967153400767,NEAR
Tag-008,Node-1,-7.337113069012945,-9.840889270920806,12.275028709946254,MID
Tag-009,Node-3,249.6591666179786,17.87164501976629,20.314900823445075,MID
Tag-010,Node-5,189.21220682857748,135.22608968143925,17.796026568864193,MID


kind,source,target
BACKHAUL,Donor-1,Node-1
BACKHAUL,Donor-1,Node-2
BACKHAUL,Donor-1,Node-3
BACKHAUL,Donor-1,Node-4
BACKHAUL,Donor-1,Node-5
UE_ASSOCIATION,Node-1,UE-1
UE_ASSOCIATION,Node-2,UE-2
UE_ASSOCIATION,Node-3,UE-3
UE_ASSOCIATION,Node-4,UE-4
UE_ASSOCIATION,Node-5,UE-5


## 7. 確認預設範例的結果

- 下方程式會自動檢查預設範例，並逐項顯示 `PASS`。
- 檢查內容包括：Tag 總數與三區數量、每個 Node 的最低數量、距離分區及連接關係。
- 也會檢查相同設定是否產生相同結果，以及不合理的 Tag 數量是否被拒絕。
- 全部執行完成後，會顯示本區塊的檢查摘要。

### 如果執行停止

- 先查看錯誤訊息，確認是否修改了 Tag 數量、區域半徑或其他設定。
- 如果改過設定，請讓檢查中的預期數值與設定一致，或先恢復預設值再試一次。
- 若生成時出現 `ValueError`，回到第 2 節核對設定限制。

- 這些檢查用來協助核對本次部署結果；不代表已計算或驗證實際無線訊號品質。


In [10]:
from dataclasses import replace, FrozenInstanceError

scenario_results = []

def passed(name):
    scenario_results.append({'情境': name, '結果': 'PASS'})

assert len(snapshot.tags) == 30 and len(snapshot.ues) == 5
assert Counter(t.zone.value for t in snapshot.tags) == {'NEAR': 10, 'MID': 13, 'FAR': 7}
assert min(Counter(t.parent_node_id for t in snapshot.tags).values()) >= 2
passed('30 Tags精確配額與每Node至少2個')
assert build_topology(config) == snapshot
assert build_topology(replace(config, seed=43)) != snapshot
passed('相同seed重現／不同seed示例有變化')
minimal = build_topology(replace(config, zone_tag_counts=(10, 0, 0)))
assert set(Counter(t.parent_node_id for t in minimal.tags).values()) == {2}
passed('10 Tags最少可行；允許空分區')
for invalid in ((9, 0, 0), (0, 0, 0)):
    try:
        build_topology(replace(config, zone_tag_counts=invalid))
    except ValueError as error:
        assert 'minimum' in str(error)
    else:
        raise AssertionError('不可行配額未拒絕')
passed('9／0 Tags明確拒絕')
assert [classify_distance(d).value for d in (10, 30, 50)] == ['NEAR', 'MID', 'FAR']
for d in (0, 50.001):
    try:
        classify_distance(d)
    except ValueError:
        pass
    else:
        raise AssertionError('非法距離未拒絕')
passed('10／30／50 m含上界，0與超界拒絕')
assert Counter(row['kind'] for row in tables['links']) == {'BACKHAUL': 5, 'UE_ASSOCIATION': 5, 'TAG_ACCESS': 30}
assert all(classify_distance(row['distance_m']) .value == row['zone'] for row in tables['tags'])
passed('歸屬表與Tag幾何一致')
state = random.getstate()
assert build_topology(config) == snapshot and state == random.getstate()
passed('生成不改全域RNG')
try:
    snapshot.config.seed = 7
except FrozenInstanceError:
    pass
else:
    raise AssertionError('快照設定可被改寫')
passed('不可變設定')
for row in scenario_results:
    print(row['結果'] + '  ' + row['情境'])
print(f'Notebook教學情境：{len(scenario_results)}項通過；完整pytest覆蓋另見執行證據。')


PASS  30 Tags精確配額與每Node至少2個
PASS  相同seed重現／不同seed示例有變化
PASS  10 Tags最少可行；允許空分區
PASS  9／0 Tags明確拒絕
PASS  10／30／50 m含上界，0與超界拒絕
PASS  歸屬表與Tag幾何一致
PASS  生成不改全域RNG
PASS  不可變設定
Notebook教學情境：8項通過；完整pytest覆蓋另見執行證據。


### 儲存檔案並查看部署圖

- 執行下方區塊後，會儲存資料與圖片，接著顯示本次部署圖。
- 輸出位置為 [artifacts/iab-topology/](artifacts/iab-topology/)。
- `deployment.png`、`deployment.svg`：部署圖，可用於查看或放入報告。
- `settings.csv`、`nodes.csv`、`tags.csv`、`links.csv`：四份表格，可用試算表工具開啟。
- `snapshot.json`：本次完整設定與部署結果。
- 同名檔案會被覆寫；需要保留舊結果時，請先複製到其他資料夾。
- 若儲存失敗，確認資料夾是否可以寫入；中途失敗可能只留下部分檔案，處理後重新匯出。


In [ ]:
output_dir = project_root / 'notebooks' / 'artifacts' / 'iab-topology'
exported = export_topology(snapshot, output_dir)
print('以下檔案皆來自本次同一快照：')
for name, path in exported.items():
    print(name, '→', path.relative_to(project_root))
display(Image(filename=str(exported['png'])))


## 8. 完成後可以做什麼？

- 打開輸出資料夾，查看圖片或用試算表工具開啟 CSV。
- 用 Tag 表確認所屬 Node、距離與區域，再與部署圖對照。
- 保留設定與 `snapshot.json`，方便日後比較不同配置。
- 需要完整操作與常見問題時，閱讀 [IAB 網路拓撲使用說明](../ReferDoc/Implementation/IABTopology/README.md)。
- 需要閱讀程式時，再看 [IAB 程式目錄](../src/iab/)；實作位於 `src/iab/`。
